---
title: "Chapter 2: Control flow and comprehensions"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/02-control-flow.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/02-control-flow.ipynb)

**DS-MLOps Python Foundations**

**Author: Anthony Faustine**

You have 2,400 student records to validate. Some scores came in as `None`. Some years of study are stored as strings. A few names have trailing spaces. You need code that checks each record, skips the broken ones, and classifies only the valid data. That is control flow.

Chapter 1 gave you the containers: lists, dicts, and strings, and the expressions to compute with them. This chapter gives those containers purpose. You will iterate over them, filter them, and transform them, and by the end, you will do it all in a single expression.

> Callout markers used throughout:
> - <i class="bi bi-key-fill"></i> **Key Concept**: the one idea that unlocks the section
> - <i class="bi bi-puzzle-fill"></i> **Activity**: a short exercise to apply the concept
> - <i class="bi bi-lightbulb-fill"></i> **Pro Tip**: a practical shortcut worth remembering

::: {.callout-note collapse="true" icon=false}
## Learning objectives

By the end of Chapter 2 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Write `if` / `elif` / `else` to branch on conditions | Sec. 1 |
| 2 | Route structured data with `match` / `case` | Sec. 2 |
| 3 | Replace manual index counters with `for`, `enumerate`, and `zip` | Sec. 3 |
| 4 | Use `while`, `break`, `continue`, and `:=` for indefinite loops | Sec. 4 |
| 5 | Replace loops with list, dict, set, and generator comprehensions | Sec. 5 |
:::

Before diving into each construct, here is the map. Python gives you four tools for controlling execution, each answering a different question about what runs and when.

![Four panels showing Python control flow: if/elif/else for branching, for loop for iteration, while loop for conditional repetition, and list comprehension for concise sequence building.](figs/control-flow-overview.svg){fig-alt="Four color-coded panels: if/elif/else (blue), for loop (green), while loop (amber), list comprehension (purple), each with a code snippet and one-line description."}

## 1. Branching with if and elif

So far every cell runs its lines from top to bottom, once, in order. An `if` statement breaks that pattern: it runs a block of code only when a condition is `True`. `elif` adds more branches. `else` is the fallback when nothing else matched.

Python evaluates each branch from top to bottom and stops at the first one that matches.

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: if / elif / else</span><br><br>
Python stops at the <em>first</em> matching branch. Subsequent <code>elif</code> and <code>else</code> blocks are skipped even if their conditions would also be true. Put the most specific condition first.
</div>

In [ ]:
def classify_grade(score: float) -> str:
    """Return letter grade for a numeric score."""
    if score >= 90:
        return "A: Excellent"
    elif score >= 80:
        return "B: Good"
    elif score >= 70:
        return "C: Satisfactory"
    elif score >= 60:
        return "D: Needs improvement"
    else:
        return "F: See instructor"

Call the function with a score from each grade band to confirm all branches work:

In [ ]:
print(classify_grade(95.0))  # A
print(classify_grade(83.5))  # B
print(classify_grade(71.0))  # C
print(classify_grade(62.0))  # D
print(classify_grade(45.0))  # F

The **ternary expression** is a one-line shorthand for a simple two-way choice. Read it as: "value if condition, otherwise other_value":

In [ ]:
score = 87.0
status = "pass" if score >= 70 else "fail"
print(f"{score} -> {status}")

## 2. Match and case

`match` / `case` was added in Python 3.10. It goes beyond simple equality: it can inspect the **shape** of data, extract values from dicts and lists in one step, and apply guard conditions with `if`.

Use `match` when you need to route a value to one of several handlers based on its structure. For a simple two-way choice, `if` / `else` is clearer.

Define a routing function that handles different types of student records. Each `case` arm matches a different dict shape:

In [ ]:
def process_record(record: dict[str, object]) -> str:
    """Route a student record to the right handler."""
    match record:
        case {"type": "enrolment", "student": s, "gpa": g} if float(str(g)) >= 3.5:
            return f"{s} enrolled: Dean's List eligible (GPA {float(str(g)):.2f})"
        case {"type": "enrolment", "student": s, "gpa": g}:
            return f"{s} enrolled (GPA {float(str(g)):.2f})"
        case {"type": "withdrawal", "student": s, "reason": r}:
            return f"{s} withdrew: {r}"
        case {"type": t}:
            return f"Unhandled record type: {t!r}"
        case _:
            return "Malformed record: missing 'type'"

Run each record through the dispatcher to see a different `case` arm trigger each time:

In [ ]:
enrolment_alice = {"type": "enrolment", "student": "Alice", "gpa": 3.8}
enrolment_bob = {"type": "enrolment", "student": "Bob", "gpa": 2.9}
withdrawal = {"type": "withdrawal", "student": "Carol", "reason": "Medical leave"}
grade_update = {"type": "grade_update"}
malformed = {"student": "Dan"}

print(process_record(enrolment_alice))
print(process_record(enrolment_bob))
print(process_record(withdrawal))
print(process_record(grade_update))
print(process_record(malformed))

### Sequence patterns in match and case

Sequence patterns match lists and tuples by position. `case [first, *rest]:` binds the first element to `first` and the remainder to `rest`. Useful for processing variable-length score records:

In [ ]:
def describe_scores(record: list) -> str:
    match record:
        case []:
            return "no scores"
        case [only]:
            return f"single score: {only}"
        case [first, second]:
            return f"two scores: {first} and {second}"
        case [first, *rest] if first >= 90:
            return f"high-starter ({first}) with {len(rest)} more scores"
        case [first, *rest]:
            return f"started at {first}, {len(rest)} more scores"

Call it with records of different lengths to trigger each case arm:

In [ ]:
print(describe_scores([]))
print(describe_scores([85]))
print(describe_scores([72, 88]))
print(describe_scores([95, 80, 77]))
print(describe_scores([60, 70, 75, 80]))

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Match on HTTP-style Status Codes</span><br><br>
<b>Goal:</b> Write a <code>describe_status(code)</code> function using <code>match/case</code> that returns a short description.
<pre style='background:#FCE8DA;padding:10px;border-radius:4px;font-size:0.9em'>describe_status(200)  -> '200 OK'
describe_status(404)  -> '404 Not Found'
describe_status(500)  -> '500 Server Error'
describe_status(301)  -> '3xx Redirect'
describe_status(999)  -> 'Unknown code'</pre>
<b>Hint:</b> Use <code>case 2xx</code> patterns are not valid. Use guard conditions instead: <code>case c if 200 <= c < 300</code>.
</div>

In [ ]:
def describe_status(code: int) -> str:
    """Return a short description for an HTTP-style status code."""
    match code:
        case _:
            return "unknown"  # TODO: replace with specific case patterns


for c in [200, 404, 500, 301, 999]:
    print(describe_status(c))

## 3. For loops

A **`for` loop** repeats a block of code once for each item in a collection. It is the primary tool for processing datasets, iterating over records, and transforming lists.

```python
for score in [78, 85, 92]:   # repeat once per score
    print(score)              # output: 78, then 85, then 92
```

The indented block (4 spaces) is the **loop body**: it runs once per item.

Python `for` loops iterate over any **iterable**. The built-ins `range()`, `enumerate()`, and `zip()` cover the most common patterns in data work.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: enumerate and zip over manual indexing</span><br><br>
<code>for i in range(len(items))</code> is a red flag. Use <code>enumerate(items)</code> when you need both the index and the value, and <code>zip(a, b)</code> when you need to step two collections in lockstep. Both are lazy: they generate pairs on demand without building a temporary list.
</div>

`range(start, stop, step)` generates integers lazily without building a list in memory. Use it to count through students, weeks, or any numbered sequence:

In [ ]:
# range(start, stop): count from 1 to WEEKS (inclusive)
WEEKS: int = 5
cumulative: int = 0

for week in range(1, WEEKS + 1):
    new_submissions = week * 12  # 12 new submissions each week
    cumulative += new_submissions
    print(f"  Week {week}/{WEEKS}  new={new_submissions}  total={cumulative}")

`enumerate()` pairs each element with its index, counting from `start=1` by default (or any integer you choose), eliminating the need for manual `i += 1` counters:

In [ ]:
# enumerate(): loop with automatic index; avoids manual counter variables
students: list[str] = ["Alice", "Carol", "Dan", "Bob"]

print("Leaderboard:")
for rank, name in enumerate(students, start=1):
    print(f"  #{rank}  {name}")

`zip()` stitches two or more iterables together element-by-element. Pairs stop when the **shortest** input is exhausted. Build a `dict` from two parallel lists using `dict(zip(keys, values))`:

In [ ]:
# zip(): iterate two or more iterables in lockstep
# strict=True raises ValueError if the iterables have different lengths
names: list[str] = ["Alice", "Bob", "Carol"]
scores: list[float] = [92.0, 74.5, 88.0]

print("Score sheet:")
for name, score in zip(names, scores, strict=True):
    grade = "pass" if score >= 70 else "fail"
    print(f"  {name:<8} {score:5.1f}  {grade}")

`dict(zip(keys, values))` is an idiomatic one-liner for building a mapping from two parallel lists:

In [ ]:
# Build a name -> score lookup from two parallel lists
names: list[str] = ["Alice", "Bob", "Carol"]
scores: list[float] = [92.0, 74.5, 88.0]

score_lookup: dict[str, float] = dict(zip(names, scores, strict=True))
print(score_lookup)

### tqdm: progress bars for long loops

When a loop processes thousands of records, you need to know how long it will take. `tqdm` wraps any iterable and displays a live progress bar with elapsed time, rate, and ETA.

Install it first:

```bash
uv add tqdm
```

In [ ]:
from tqdm import tqdm

# Wrap any iterable with tqdm() - the loop body is unchanged
scores: list[float] = []
for i in tqdm(range(1_000), desc="Simulating scores", unit="rec"):
    scores.append(50 + (i % 50))  # dummy computation

print(f"Generated {len(scores)} scores, mean = {sum(scores) / len(scores):.1f}")

# tqdm also works with enumerate and zip
labels: list[str] = ["pass" if s >= 70 else "fail" for s in tqdm(scores, desc="Labelling", leave=False)]
print(f"pass rate: {labels.count('pass') / len(labels):.1%}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2: cohort score summary</span><br><br>
<b>Goal:</b> Print a formatted score summary using <code>enumerate</code> and <code>zip</code>.<br><br><b>Setup:</b><br><code>students = ["Alice", "Bob", "Carol", "Dan", "Eve"]</code><br><code>midterm  = [82.0, 65.0, 91.0, 74.0, 88.0]</code><br><code>final    = [79.0, 70.0, 94.0, 68.0, 85.0]</code><br><br><b>Steps:</b><br>1. Use <code>zip</code> to pair <code>midterm</code> and <code>final</code> for each student.<br>2. Use <code>enumerate</code> to add a rank number starting from 1.<br>3. Compute the average of midterm and final for each student.<br>4. Print: <code>#1 Alice  mid=82.0  final=79.0  avg=80.5</code><br>5. Print the name of the student with the highest average.<br></div>

## 4. While loops, break, and continue

A **`while` loop** repeats a block as long as a condition is `True`. Unlike `for` (which iterates a fixed collection), `while` runs an **indefinite** number of times until either the condition becomes `False` or a `break` statement is hit.

```python
pending = 24
while pending > 0:    # keep processing until the queue is empty
    pending -= 3      # process 3 records per batch
```

Use `while` when you do not know in advance how many iterations you need: retrying a failing operation, reading until end of file, or consuming a queue one batch at a time.

- `break` exits the loop immediately
- `continue` skips the rest of the current iteration and moves to the next

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: while for indefinite loops, for for known collections</span><br><br>
Use <code>for</code> when you know the collection to iterate. Use <code>while</code> when you don't know how many iterations you need: retrying until success, reading until end of file, or consuming a queue until it is empty. Every <code>while</code> loop needs either a condition that eventually becomes <code>False</code> or an explicit <code>break</code>; otherwise it runs forever.
</div>

Process a submission queue in batches until it is empty or the processing budget is exhausted:

In [ ]:
# while: consume a queue in fixed-size batches
queue: list[str] = [f"student_{i:03d}" for i in range(1, 18)]  # 17 submissions
processed: int = 0
BATCH: int = 5

while queue:
    batch = queue[:BATCH]
    queue = queue[BATCH:]
    processed += len(batch)
    print(f"  Processed batch of {len(batch):2d}  ({len(queue)} remaining)")

print(f"Total processed: {processed}")

### break and continue
`break` exits the innermost loop immediately. Use it when a sentinel value or error condition means further iteration is pointless:

In [ ]:
# break: exit the loop immediately when a sentinel is found
readings: list[float | None] = [36.5, 36.9, 37.4, None, 38.1, 37.8]
clean: list[float] = []

for r in readings:
    if r is None:
        print("Sensor error : stopping collection")
        break
    clean.append(r)

print(f"Clean readings: {clean}")

`continue` skips the rest of the current iteration and jumps to the next one. Ideal for filtering bad data without a nested `if/else`. The `else` clause on a loop runs only if no `break` occurred:

In [ ]:
# continue: skip the rest of this iteration and move to the next
raw: list[object] = [85.0, "n/a", None, 92.0, "", 78.5, -1.0, 95.0]
valid: list[float] = []

for item in raw:
    if not isinstance(item, int | float) or float(str(item)) < 0:
        continue  # skip bad items
    valid.append(float(str(item)))

print(f"Valid scores: {valid}")

A loop can have an `else` clause that runs only when the loop **was not** exited by `break`. It is the clean way to handle "found nothing" after a search:

In [ ]:
required_fields: list[str] = ["name", "gpa", "major"]
record: dict[str, str] = {"name": "Alice", "gpa": "3.95", "major": "CS"}

for field in required_fields:
    if field not in record:
        print(f"Missing field: {field}")
        break
else:
    print("Record is complete")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3: find the first failing student</span><br><br>
<b>Goal:</b> Process a list of scores and stop as soon as you find the first score below 60.<br><br><b>Setup:</b><br><code>scores = [82.0, 91.0, 74.0, 65.0, 55.0, 88.0, 42.0]</code><br><br><b>Steps:</b><br>1. Use a <code>while</code> loop with an index variable.<br>2. If the current score is below 60, print the index and score, then <code>break</code>.<br>3. After the loop, print how many records were checked before the fail was found.<br><b>Expected output:</b><br><code>First fail at index 4: 55.0</code><br><code>Checked 5 records before finding it</code>
</div>

### The walrus operator

Python 3.8 added `:=`, called the **walrus operator** (it looks like a walrus on its side). It assigns a value and evaluates it in the same expression.

Its natural home is a `while` loop that needs to read and test in one step:

```python
# Without walrus: assign before the loop, update at the end
line = f.readline()
while line:
    process(line)
    line = f.readline()     # easy to forget this update

# With walrus: read and check in one expression
while line := f.readline():
    process(line)
```

In [ ]:
import io

# Simulate a file of student records
data = io.StringIO("Alice,92\nBob,74\nCarol,88\n")

# walrus: read a line and check it is non-empty in one expression
while line := data.readline():
    name, score = line.strip().split(",")
    print(f"  {name}: {score}")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: walrus := assigns and returns</span><br><br>
<code>:=</code> assigns to the left-hand variable and returns the assigned value. Use it when you would otherwise write "assign, then check" at the top of a loop and "assign again" at the bottom. Overuse hurts readability; save it for the <code>while (value := read_something()):</code> pattern.
</div>

## 5. Comprehensions

A **comprehension** builds a new collection by transforming or filtering an existing one, all in a single expression. It replaces the verbose `for` + `.append()` pattern:

```python
# Loop version (3 lines):
squares = []
for n in range(5):
    squares.append(n ** 2)   # [0, 1, 4, 9, 16]

# Comprehension (1 line, identical result):
squares = [n ** 2 for n in range(5)]
```

Comprehensions are faster than equivalent loops and are considered idiomatic Python.

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Concise, Readable Collection Construction</span><br><br>
Comprehensions build new collections by <b>transforming or filtering</b> an iterable in a single expression. They are faster than equivalent <code>for</code> + <code>.append()</code> loops and are idiomatic Python.

<table style='margin-top:8px;border-collapse:collapse'>
<tr><td style='padding:4px 12px;font-family:monospace'>[expr for x in it if cond]</td><td style='padding:4px 12px'>list</td></tr>
<tr><td style='padding:4px 12px;font-family:monospace'>{k: v for x in it if cond}</td><td style='padding:4px 12px'>dict</td></tr>
<tr><td style='padding:4px 12px;font-family:monospace'>{expr for x in it if cond}</td><td style='padding:4px 12px'>set</td></tr>
<tr><td style='padding:4px 12px;font-family:monospace'>(expr for x in it if cond)</td><td style='padding:4px 12px'>generator (lazy, no list in memory)</td></tr>
</table>
</div>

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Comprehensions are for simple transformations; loops are for complex logic</span><br><br>
A list comprehension <code>[f(x) for x in items if cond(x)]</code> is readable when the filter and transform are each one expression. When you need multiple steps, early returns, or side effects, write a <code>for</code> loop. Nested comprehensions beyond two levels hurt readability: that's the signal to refactor.
</div>

Transform a list with a comprehension: min-max normalise scores to the range [0, 1]:

In [ ]:
raw_scores: list[float] = [78.0, 85.5, 92.0, 88.5, 95.0, 67.0, 81.0]

lo, hi = min(raw_scores), max(raw_scores)
normed: list[float] = [(s - lo) / (hi - lo) for s in raw_scores]
print(f"Normalised: {[round(n, 2) for n in normed]}")

Filter with a comprehension: keep only the passing scores:

In [ ]:
raw_scores: list[float] = [78.0, 85.5, 92.0, 88.5, 95.0, 67.0, 81.0]

passing: list[float] = [s for s in raw_scores if s >= 70]
print(f"Passing: {passing}")

Combine filter and transform in one comprehension: attach a pass/fail label to each score:

In [ ]:
raw_scores: list[float] = [78.0, 85.5, 92.0, 88.5, 95.0, 67.0, 81.0]

labels: list[str] = [f"{s:.0f} (pass)" if s >= 70 else f"{s:.0f} (FAIL)" for s in raw_scores]
print(f"Labelled: {labels}")

A two-clause comprehension flattens a nested collection. Read `[s for batch in batches for s in batch]` left-to-right: "outer loop, inner loop, collect `s`":

In [ ]:
# Flatten a nested structure with a two-clause comprehension
batches: list[list[float]] = [[85.0, 91.0], [74.0, 88.5], [95.0, 79.0]]
flat: list[float] = [s for batch in batches for s in batch]
print(f"Flattened : {flat}")

### Dict, set, and generator comprehensions
The `[...]` syntax extends to dicts (`{k: v for ...}`), sets (`{expr for ...}`), and lazy generators (`(expr for ...)`):

In [ ]:
students: list[dict[str, object]] = [
    {"name": "Alice", "score": 92.0, "major": "CS"},
    {"name": "Bob", "score": 74.5, "major": "Math"},
    {"name": "Carol", "score": 88.0, "major": "CS"},
    {"name": "Dan", "score": 61.0, "major": "Physics"},
]

# Dict comprehension: build a name -> score lookup
score_lookup: dict[str, float] = {str(s["name"]): float(str(s["score"])) for s in students}
print(f"Lookup : {score_lookup}")

# Dict comprehension with filter: honours students only
honours: dict[str, float] = {str(s["name"]): float(str(s["score"])) for s in students if float(str(s["score"])) >= 80}
print(f"Honours: {honours}")

A **set comprehension** `{expr for x in it}` automatically deduplicates the results:

In [ ]:
students: list[dict[str, object]] = [
    {"name": "Alice", "score": 92.0, "major": "CS"},
    {"name": "Bob", "score": 74.5, "major": "Math"},
    {"name": "Carol", "score": 88.0, "major": "CS"},
    {"name": "Dan", "score": 61.0, "major": "Physics"},
]

# Set comprehension: unique majors (duplicates removed automatically)
majors: set[str] = {str(s["major"]) for s in students}
print(f"Majors: {sorted(majors)}")

A **generator expression** `(expr for x in it)` computes values **lazily**: it uses O(1) memory regardless of input size. Pass one directly to `sum()`, `any()`, or `all()` to avoid building a temporary list:

In [ ]:
students: list[dict[str, object]] = [
    {"name": "Alice", "score": 92.0, "major": "CS"},
    {"name": "Bob", "score": 74.5, "major": "Math"},
    {"name": "Carol", "score": 88.0, "major": "CS"},
    {"name": "Dan", "score": 61.0, "major": "Physics"},
]

total: float = sum(float(str(s["score"])) for s in students)
print(f"Mean          : {total / len(students):.1f}")

In [ ]:
students: list[dict[str, object]] = [
    {"name": "Alice", "score": 92.0, "major": "CS"},
    {"name": "Bob", "score": 74.5, "major": "Math"},
    {"name": "Carol", "score": 88.0, "major": "CS"},
    {"name": "Dan", "score": 61.0, "major": "Physics"},
]

# any() and all() short-circuit: they stop as soon as the answer is known
any_fail: bool = any(float(str(s["score"])) < 70 for s in students)
all_pass: bool = all(float(str(s["score"])) >= 60 for s in students)
print(f"Any fail (<70): {any_fail}")
print(f"All pass (>=60): {all_pass}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Cohort Score Report</span><br><br>
<b>Goal:</b> Using a single comprehension for each, produce the outputs below from <code>records</code>.
<pre style='background:#FCE8DA;padding:10px;border-radius:4px;font-size:0.9em'>records = [
    {'name': 'Alice', 'scores': [88, 92, 85]},
    {'name': 'Bob',   'scores': [62, 70, 58]},
    {'name': 'Carol', 'scores': [91, 95, 89]},
]

# 1. List of averages (one float per student)
averages = [82.33, 63.33, 91.67]

# 2. Dict mapping name -> average (rounded to 2 dp)
avg_map = {'Alice': 88.33, 'Bob': 63.33, 'Carol': 91.67}

# 3. Set of unique student names who scored >= 80 average
top = {'Alice', 'Carol'}</pre>
</div>

In [ ]:
records: list[dict[str, object]] = [
    {"name": "Alice", "scores": [88, 92, 85]},
    {"name": "Bob", "scores": [62, 70, 58]},
    {"name": "Carol", "scores": [91, 95, 89]},
]

# TODO: 1. list of averages
averages: list[float] = ...

# TODO: 2. name -> average dict
avg_map: dict[str, float] = ...

# TODO: 3. set of names with average >= 80
top: set[str] = ...

print(f"averages: {averages}")
print(f"avg_map : {avg_map}")
print(f"top     : {top}")

`itertools.batched()` (Python 3.12) splits any iterable into fixed-size chunks without loading everything into memory. It replaces manual sliding-window loops in data pipelines.

In [ ]:
from itertools import batched

student_ids = list(range(1, 22))  # 21 students

for batch_num, batch in enumerate(batched(student_ids, 5), start=1):
    print(f"Batch {batch_num}: {list(batch)}")

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: batched() for Chunked Processing</span><br><br>
<code>batched(iterable, n)</code> yields tuples of at most <code>n</code> items. The last batch may be shorter. Use it for API rate-limiting, mini-batch gradient descent, or processing large CSV files in chunks: <code>for rows in batched(csv_reader, 1000): process(rows)</code>.
</div>

## Capstone: Processing a student record set

This activity brings together everything from Chapters 1 and 2. You will read a list of student records that simulates a CSV file, classify each student, search for a specific one, and build a cohort summary using a comprehension.

The four control flow constructs work together here: `if`/`elif` to classify, `for` to iterate, `while` to search, and a comprehension to summarize.

**Step 1:** Set up the records. `csv.DictReader` and `io.StringIO` let you parse a CSV string without a file on disk:

In [ ]:
import csv
import io

CSV_DATA = (
    "name,score,year\nAlice,92.5,2\nBob,74.0,3\nCarol,88.0,1\nDan,61.5,2\nEve,95.0,3\nFiona,55.0,1\nGeorge,78.0,2\n"
)

reader = csv.DictReader(io.StringIO(CSV_DATA))
records: list[dict[str, object]] = [
    {"name": row["name"], "score": float(row["score"]), "year": int(row["year"])} for row in reader
]
print(f"Loaded {len(records)} student records")
print(records[0])

**Step 2:** Define a grading function using `if`/`elif`/`else`:

In [ ]:
def classify(score: float) -> str:
    if score >= 90:
        return "distinction"
    elif score >= 70:
        return "pass"
    else:
        return "fail"

**Step 3:** Use a `for` loop to classify every student and print a formatted table:

In [ ]:
print(f"{'Name':<10} {'Score':>6}  {'Grade':<12}  Year")
print("-" * 38)
for rec in records:
    grade = classify(float(str(rec["score"])))
    print(f"{str(rec['name']):<10} {float(str(rec['score'])):>6.1f}  {grade:<12}  {rec['year']}")

**Step 4:** Use a `while` loop to find the first distinction student without scanning the whole list:

In [ ]:
idx: int = 0
first_distinction: dict[str, object] | None = None

while idx < len(records):
    rec = records[idx]
    if float(str(rec["score"])) >= 90:
        first_distinction = rec
        break
    idx += 1

if first_distinction:
    print(f"First distinction: {first_distinction['name']} ({first_distinction['score']})")
else:
    print("No distinction students found")

**Step 5:** Use comprehensions to build the cohort summary in three lines:

In [ ]:
grades: list[str] = [classify(float(str(r["score"]))) for r in records]

grade_counts: dict[str, int] = {g: grades.count(g) for g in ["distinction", "pass", "fail"]}
print(f"Grade distribution: {grade_counts}")

In [ ]:
top_students: list[str] = [str(r["name"]) for r in records if float(str(r["score"])) >= 80]
print(f"Top students (>=80): {top_students}")

In [ ]:
mean_score: float = sum(float(str(r["score"])) for r in records) / len(records)
print(f"Cohort mean: {mean_score:.1f}")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> What you used in this capstone</span><br><br>
<ul>
<li><b>if / elif / else:</b> the <code>classify()</code> function routes each score to a grade band</li>
<li><b>for loop:</b> iterating over every record to print the full table</li>
<li><b>while + break:</b> stopping the search as soon as the first distinction is found</li>
<li><b>Comprehensions:</b> building grade counts, top-student lists, and a running total each in one expression</li>
<li><b>Walrus (:=):</b> reading records line by line in the earlier section</li>
</ul>
These four constructs cover nearly every control flow situation you will encounter in data work.
</div>

## Further Reading

| Resource | Why it matters |
|---|---|
| [PEP 636: Structural Pattern Matching](https://peps.python.org/pep-0636/) | Official tutorial for `match`/`case`, with worked examples from the Python core team |
| Ramalho, L. (2022). *Fluent Python*, 2nd ed. O'Reilly. | Chapter 10 covers pattern matching in depth, including class patterns and guards |
| [Real Python: Python `for` Loops](https://realpython.com/python-for-loop/) | Clear treatment of `enumerate`, `zip`, and the iterator protocol behind every loop |
| [Real Python: List Comprehensions](https://realpython.com/list-comprehension-python/) | When to use comprehensions vs explicit loops, and how to avoid making them unreadable |


## Summary

| Concept | Key rule |
|---|---|
| `if` / `elif` / `else` | Evaluate top to bottom; stop at the first match |
| `match` / `case` | Structural pattern matching on values, dicts, and lists |
| `enumerate` / `zip` | Always prefer these over manual index counters |
| `while` / `break` / `continue` | For indefinite loops, early exit, and skipping bad data |
| `:=` walrus | Assign and test in one expression; natural in `while line := f.readline():` |
| Comprehensions | `[expr for x in it if cond]`; use generators `(...)` inside `sum()` / `any()` / `all()` |

**Next:** `03-functions.ipynb`: functions, default arguments, *args/**kwargs, and docstrings., default arguments, `*args`/`**kwargs`, and writing docstrings.